# LangGraph 简明教程

LangGraph 是基于 LangChain 的**状态机图框架**，核心概念：
- **State**：节点间共享的状态（dict）
- **Node**：处理状态的函数
- **Edge**：节点间的连接（普通边 / 条件边）
- **Graph**：把上面三者组装起来

## 0. 安装依赖

In [1]:
%cd /data/projects_ING/ollama-python


/data/projects_ING/ollama-python


In [ ]:
# %pip install -q langgraph langchain-ollama
!uv add ipykernel langgraph langchain-ollama 


## 1. 最简单的图：单节点

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

# 1. 定义 State
class State(TypedDict):
    message: str

# 2. 定义 Node（函数，接收 state，返回更新的字段）
def hello_node(state: State) -> State:
    return {"message": f"Hello, {state['message']}!"}

# 3. 构建图
graph = StateGraph(State)
graph.add_node("hello", hello_node)
graph.set_entry_point("hello")   # 入口节点
graph.add_edge("hello", END)     # 连接到结束
app = graph.compile()

# 4. 运行
result = app.invoke({"message": "World"})
print(result)


## 2. 多节点串行

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class State(TypedDict):
    value: int

def double(state: State) -> State:
    return {"value": state["value"] * 2}

def add_ten(state: State) -> State:
    return {"value": state["value"] + 10}

graph = StateGraph(State)
graph.add_node("double", double)
graph.add_node("add_ten", add_ten)

graph.set_entry_point("double")
graph.add_edge("double", "add_ten")  # double → add_ten → END
graph.add_edge("add_ten", END)
app = graph.compile()

print(app.invoke({"value": 5}))  # 5*2+10 = 20


## 3. 条件边（分支）

根据状态决定走哪条路

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END

class State(TypedDict):
    value: int

def check(state: State) -> Literal["big", "small"]:
    return "big" if state["value"] >= 10 else "small"

def big_node(state: State) -> State:
    return {"value": state["value"] - 10}

def small_node(state: State) -> State:
    return {"value": state["value"] + 100}

graph = StateGraph(State)
graph.add_node("big", big_node)
graph.add_node("small", small_node)

# 条件边：从 START 根据 check() 结果路由
graph.set_conditional_entry_point(check, {"big": "big", "small": "small"})
graph.add_edge("big", END)
graph.add_edge("small", END)
app = graph.compile()

print(app.invoke({"value": 15}))  # big → 15-10=5
print(app.invoke({"value": 3}))   # small → 3+100=103


## 4. 循环（Loop）

条件边指回自身，实现 while 循环效果

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END

class State(TypedDict):
    count: int

def increment(state: State) -> State:
    print(f"count = {state['count']}")
    return {"count": state["count"] + 1}

def should_continue(state: State) -> Literal["continue", "stop"]:
    return "continue" if state["count"] < 5 else "stop"

graph = StateGraph(State)
graph.add_node("increment", increment)
graph.set_entry_point("increment")

# 条件边：continue → 回到 increment，stop → END
graph.add_conditional_edges("increment", should_continue, {
    "continue": "increment",
    "stop": END
})
app = graph.compile()
app.invoke({"count": 0})


## 5. 接入 Ollama（实际 AI 对话）

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, BaseMessage
import operator

MODEL = "gemma4"  # 与 bridge_config.json 保持一致
llm = ChatOllama(model=MODEL)

# Annotated + operator.add 让 messages 自动追加而非覆盖
class State(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

def call_llm(state: State) -> State:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

graph = StateGraph(State)
graph.add_node("llm", call_llm)
graph.set_entry_point("llm")
graph.add_edge("llm", END)
app = graph.compile()

result = app.invoke({"messages": [HumanMessage(content="用一句话解释什么是LangGraph")]})
print(result["messages"][-1].content)


## 6. 多轮对话（保留历史）

在循环图中持续追加消息，实现真正的对话

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, BaseMessage
import operator

llm = ChatOllama(model="gemma4")

class State(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

def call_llm(state: State) -> State:
    return {"messages": [llm.invoke(state["messages"])]}

def has_more_input(state: State) -> Literal["continue", "stop"]:
    # 实际项目中可检查 tool_calls 或外部信号
    return "stop"  # 这里简化为单轮

graph = StateGraph(State)
graph.add_node("llm", call_llm)
graph.set_entry_point("llm")
graph.add_conditional_edges("llm", has_more_input, {"continue": "llm", "stop": END})
app = graph.compile()

# 模拟多轮：手动追加历史
history = []
for user_input in ["你好", "刚才我说了什么？"]:
    history.append(HumanMessage(content=user_input))
    result = app.invoke({"messages": history})
    reply = result["messages"][-1]
    history.append(reply)
    print(f"User: {user_input}")
    print(f"AI:   {reply.content}\n")


## 7. 可视化图结构

In [ ]:
# 需要安装: pip install grandalf
# 或直接用 ASCII 打印

# 直接打印节点和边的文本描述
g = app.get_graph()
print("节点:", [n for n in g.nodes])
print("边:", [(e.source, e.target) for e in g.edges])


## 总结

| 概念 | 对应代码 |
|------|----------|
| State | `TypedDict` |
| Node | 普通函数，返回 state 更新 |
| 普通边 | `add_edge(a, b)` |
| 条件边 | `add_conditional_edges(node, fn, map)` |
| 循环 | 条件边指回自身 |
| 消息追加 | `Annotated[list, operator.add]` |

下一步可以结合 `main.py` 中的 MCP 调用，把 `chat_with_mcp` 封装成 LangGraph 的一个节点。